# Judge evaluation (Kaggle version)
Runs the entailment judge (Qwen2.5-7B) on BOTH hand-labelled rounds:
- round 1: the 49 facts the judge and the rules were designed on (mostly English facts)
- round 2: 41 NEW Spanish facts, labelled with the written rubric (held-out: nothing was tuned on them)

**Before running:** Session options → **GPU T4 x2**, **Internet: On**. About 5-10 min.

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('newsroom-agents'):
    !git clone https://github.com/Juanangara/newsroom-agents.git
%cd /kaggle/working/newsroom-agents
!git pull -q
!pip install -q pydantic feedparser trafilatura googlenewsdecoder bitsandbytes accelerate

In [ ]:
import sys, json, torch
sys.path.insert(0, '/kaggle/working/newsroom-agents/src')
print('GPUs:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
from newsroom import HFLocalLLM
from newsroom.label_eval import evaluate_judge, print_report

llm = HFLocalLLM('Qwen/Qwen2.5-7B-Instruct')

In [ ]:
import os
out = '/kaggle/working/out'
os.makedirs(out, exist_ok=True)
ROUNDS = {
    'round1': ('results/benchmark_diag.json', 'results/labeling/labels_filled.csv'),
    'round2_es': ('results/benchmark_diag_es.json', 'results/labeling/labels_filled_es.csv'),
}
for name, (ev, lab) in ROUNDS.items():
    print()
    print('=' * 70)
    print(name)
    print('=' * 70)
    res = evaluate_judge(llm, ev, lab)
    print_report(res)
    json.dump(res, open(f'{out}/judge_eval_{name}.json', 'w'), ensure_ascii=False, indent=1)


Output files: Output tab → `out/judge_eval_round1.json` and `out/judge_eval_round2_es.json` (every disagreement with your labels is listed).